# Combine Trend Signals Explorer

Shows how `combine_trend_signals.py` pools raw counts across retailers and divides by the grand total to produce one proportion score per feature value.

In [10]:
import json
import math
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path('../../').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'trndly' / 'pipelines' / 'training' / 'synthetic_data'

In [11]:
DEFAULT_MISSING_SCORE = 0.05

retailer_files = sorted(DATA_DIR.glob('trend_signals_*.csv'))
# exclude the combined output file itself
retailer_files = [f for f in retailer_files if f.name != 'trend_signals.csv']

frames = {}
total_items = {}

for csv_path in retailer_files:
    retailer = csv_path.stem.replace('trend_signals_', '')
    df = pd.read_csv(csv_path)
    frames[retailer] = df

    meta_path = csv_path.with_name(csv_path.stem + '_meta.json')
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        total_items[retailer] = meta.get('total_items', 1.0)
        print(f'{retailer:<20} {len(df):>3} rows   total_items={total_items[retailer]}')
    else:
        total_items[retailer] = 1.0
        print(f'{retailer:<20} {len(df):>3} rows   total_items=1.0 (NO META FILE — re-run scraper to fix)')

american_eagle        27 rows   total_items=560
gap                   27 rows   total_items=926
hollister             27 rows   total_items=646
uniqlo                27 rows   total_items=2371


In [ ]:
pivot_frames = []
for retailer, df in frames.items():
    pf = df.set_index(['feature_type', 'feature_value'])[['current']]
    pf.columns = [retailer]
    pivot_frames.append(pf)

side_by_side = pd.concat(pivot_frames, axis=1).reset_index()
side_by_side

,feature_type,feature_value,american_eagle,gap,hollister,uniqlo
0,color,black,0.019643,0.502160,0.687307,0.118094
1,color,white,0.021429,0.381210,0.591331,0.097849
2,color,blue,0.021429,0.763499,0.286378,0.090257
3,color,red,0.235714,0.192225,0.024768,0.031632
4,color,green,0.021429,0.177106,0.105263,0.056938
5,color,beige,0.158929,0.160907,0.156347,0.056938
6,color,pink,0.019643,0.099352,0.270898,0.023197
7,color,gray,0.021429,0.090713,0.555728,0.093210
8,color,navy,0.050000,0.166307,0.424149,0.070434
9,color,brown,0.019643,0.220302,0.386997,0.046394


In [4]:
grand_total = sum(total_items.values())
print(f'Grand total items across all retailers: {grand_total:,.0f}')
for r, n in total_items.items():
    print(f'  {r:<20} {n:>6,.0f} items  ({100*n/grand_total:.1f}% of grand total)')

Grand total items across all retailers: 4,503
  american_eagle          560 items  (12.4% of grand total)
  gap                     926 items  (20.6% of grand total)
  hollister               646 items  (14.3% of grand total)
  uniqlo                2,371 items  (52.7% of grand total)


In [5]:
def is_real(score):
    return not math.isclose(float(score), DEFAULT_MISSING_SCORE, rel_tol=0.0, abs_tol=1e-9)

rows = []
for _, row in side_by_side.iterrows():
    ft  = row['feature_type']
    fv  = row['feature_value']
    raw_counts = {}
    for retailer in frames:
        score = row.get(retailer, DEFAULT_MISSING_SCORE)
        if pd.isna(score):
            score = DEFAULT_MISSING_SCORE
        raw = float(score) * total_items[retailer] if is_real(score) else 0.0
        raw_counts[retailer] = raw

    total_raw = sum(raw_counts.values())
    combined  = round(total_raw / grand_total, 6) if grand_total > 0 else DEFAULT_MISSING_SCORE
    combined  = max(combined, DEFAULT_MISSING_SCORE)

    result = {'feature_type': ft, 'feature_value': fv}
    result.update({f'{r}_raw': round(v, 1) for r, v in raw_counts.items()})
    result['total_raw'] = round(total_raw, 1)
    result['combined_score'] = combined
    rows.append(result)

combined_df = pd.DataFrame(rows)
combined_df

,feature_type,feature_value,american_eagle_raw,gap_raw,hollister_raw,uniqlo_raw,total_raw,combined_score
0,color,black,11.0,465.0,444.0,280.0,1200.0,0.266489
1,color,white,12.0,353.0,382.0,232.0,979.0,0.217411
2,color,blue,12.0,707.0,185.0,214.0,1118.0,0.248279
3,color,red,132.0,178.0,16.0,75.0,401.0,0.089052
4,color,green,12.0,164.0,68.0,135.0,379.0,0.084166
5,color,beige,89.0,149.0,101.0,135.0,474.0,0.105263
6,color,pink,11.0,92.0,175.0,55.0,333.0,0.073951
7,color,gray,12.0,84.0,359.0,221.0,676.0,0.150123
8,color,navy,0.0,154.0,274.0,167.0,595.0,0.132134
9,color,brown,11.0,204.0,250.0,110.0,575.0,0.127693


In [6]:
for feature_type, group in combined_df.groupby('feature_type'):
    print(f'\n── {feature_type} ──')
    ranked = group[['feature_value', 'combined_score']].sort_values('combined_score', ascending=False)
    for _, r in ranked.iterrows():
        bar = '█' * int(r['combined_score'] * 80)
        print(f"  {r['feature_value']:<18} {r['combined_score']:.4f}  {bar}")


── category ──
  tops               0.2461  ███████████████████
  pants              0.0968  ███████
  dress              0.0895  ███████
  outerwear          0.0513  ████
  shorts             0.0500  ████
  skirt              0.0500  ████
  shoes              0.0500  ████
  accessories        0.0500  ████

── color ──
  black              0.2665  █████████████████████
  blue               0.2483  ███████████████████
  white              0.2174  █████████████████
  gray               0.1501  ████████████
  navy               0.1321  ██████████
  brown              0.1277  ██████████
  beige              0.1053  ████████
  red                0.0891  ███████
  green              0.0842  ██████
  pink               0.0740  █████
  purple             0.0500  ████

── material ──
  cotton             0.2794  ██████████████████████
  denim              0.1519  ████████████
  polyester          0.0562  ████
  linen              0.0500  ████
  silk               0.0500  ████
  wool           

In [9]:
trend_signals = pd.read_csv(DATA_DIR / 'trend_signals.csv')
trend_signals

,feature_type,feature_value,current
0,category,accessories,0.050000
1,category,dress,0.089496
2,category,outerwear,0.051299
3,category,pants,0.096824
4,category,shoes,0.050000
5,category,shorts,0.050000
6,category,skirt,0.050000
7,category,tops,0.246058
8,color,beige,0.105263
9,color,black,0.266489
